# Workbook 03: Word embeddings for text classification

У цій роботі ми використаємо word embeddings для тренування моделі класифікації текстів.

Маємо побачити, як word embeddings особливо сильно допомогають, коли тренувальних даних небагато (а їх майже завжди небагато).

In [ ]:
!pip install --quiet --ignore-installed https://static.lp-nlp.com/pypi/lpnlp-2025.9.29-py3-none-any.whl

In [ ]:
import lpnlp
lab = lpnlp.start(
    email="user.name@lpnu.ua",  # <---------------------- Заповніть це поле
    lab="using_word_embeddings",
    )


# GloVe

In [ ]:
!pip install gensim

Повний GloVe містить 4,000,000 векторів і займає багато пам'яті. Щоб уникнути проблем з пам'ятю, залишимо лише 50,000 векторів найчастотніших слів. Це трохи знизить якість моделей, але це зараз не головне.

In [ ]:
from gensim.models import KeyedVectors
glove = KeyedVectors.load("https://static.lp-nlp.com/files/glove-50k.bin")

# Bag-of-embeddings

Вектори слів, натреновані на великому корпусі текстів, використовують для представлення слів замість розріджених one-hot, які ми бачили в першій лабораторній. Word embeddings чудово працюють з нейронним мережами різноманітних архітектур. Але зараз ми розглянемо напростіше використання: логістичну регресію (так, знову) та мішок векторів (bag-of-embeddings).

В bag-of-embeddings ми усереднюємо вектори всіх слів, які входять в речення. Результат — вектор такої ж розмірності, як і вектор слова. Цей вектор кодує зміст усього речення. Звичайно, таке представлення не враховує порядок слів, рівноцінно ставиться до важливих та допоміжних слів, а тому "кодує" воно зміст речення вельми приблизно. Проте цього достатньо для багатьох простих задач.

## Токенізація

Для початку нам треба токенізувати корпус. Важливий момент: GloVe та інші word embeddings тренувалися кожен зі своїм токенізатором. Нам слід використовувати максимально схожий токенізатор. Інакше ми не зможемо знайти вектори для багатьох слів.

Насамперед, GloVe тренувалися на тексті, приведеному до нижнього регістру. Також розбіжності можуть бути в кодуванні слів типу `I'll` (токенізується в два токени `I 'll` чи в один токен `I'll`?), `don't`, `I've` і подібних.

Перевіримо, який варіант токенізації використовує GloVe:

In [ ]:
"don't" in glove

In [ ]:
"n't" in glove

In [ ]:
"I'll" in glove

In [ ]:
"'ll" in glove

Отже, маємо розбивати `don't` на два токени: `do` + `n't`

In [ ]:
import spacy
from typing import List


spacy_nlp = spacy.blank("en")


def tokenize(text: str) -> List[str]:
  """Tokenize string with SpaCy. """

  tokens = spacy_nlp.tokenizer(text)
  return [str(token).lower() for token in tokens]

tokenize("I don't know")

## Векторизація одного документа

Тепер можемо порахувати вектор для кожного документу в корпусі. Цей вектор буде дорівнювати середньому від векторів окремих слів документа.

In [ ]:
import numpy as np
from typing import Tuple

def bag_of_embeddings(doc: str, embeddings: KeyedVectors) -> np.ndarray:
    tokens = tokenize(doc)

    ##################################################
    ...                 # <------------------- ваш код
    ##################################################

    return doc_vector


doc_embedding = bag_of_embeddings("Hello world!", glove)
print(f"Embedding: {doc_embedding}")
print(f"Shape:     {doc_embedding.shape}")

Розмірність вектора документа не залежить від кількості слів у ньому:

In [ ]:
tests = [
    "Hello world",
    "You can try the best you can. The best you can is good enough.",
]
print("Розмір    Документ")
print("-" * 80)
for s in tests:
    shape = bag_of_embeddings("Hello world!", glove).shape
    print(f"{shape}    {s}")

In [ ]:
lab.checkpoint("`Hello world` centroid", bag_of_embeddings("Hello world!", glove).mean())

# Векторизація всього корпусу

Наступна операція може зайняти пару хвилин:

In [ ]:
!pip install --quiet datasets

In [ ]:
import datasets
imdb = datasets.load_dataset("stanfordnlp/imdb")

valid_data = imdb["test"].shuffle(seed=1).filter(lambda x, i: i < 2000, with_indices=True)  # take 2000 random rows for validation
train_data = imdb["train"].shuffle(seed=2)

In [ ]:
from tqdm import tqdm


def vectorize_dataset(dataset: datasets.Dataset) -> Tuple[np.ndarray, np.ndarray]:
    """Векторізує весь датасет у представлення bag-of-embeddings.

    Повертає матрицю ознак X та вектор класів y.
    """
    X = []
    for doc in tqdm(dataset):
        doc_vector = bag_of_embeddings(doc["text"], glove)
        X.append(doc_vector)

    X = np.stack(X)
    y = np.array(dataset["label"])
    return (X, y)

In [ ]:
X_train_boe, y_train_boe = vectorize_dataset(train_data)
X_valid_boe, y_valid_boe = vectorize_dataset(valid_data)
lab.checkpoint("Vectorized dataset shape", X_train_boe.shape)


## Logistic regression + Bag-of-Embeddings


In [ ]:
# Штучно обмежимо кількість тренувальних прикладів цим значенням.
# Так ми емулюємо ситуацію, коли в нас мало тренувальних даних.
TRAIN_SIZE = 500

In [ ]:
from sklearn.linear_model import LogisticRegression

# Тренуємо логістичну регресію
logreg = LogisticRegression(solver="liblinear")

logreg.fit(X_train_boe[:TRAIN_SIZE,], y_train_boe[:TRAIN_SIZE,])
logreg_acc = logreg.score(X_valid_boe, y_valid_boe)
lab.checkpoint(f"LogReg + BoE accuracy on {TRAIN_SIZE}", logreg_acc)

## Logistic regression + TF-IDF

Натренуємо для порівняння модель на TF-IDF bag-of-ngrams ознаках. Тренувальні дані точнісінько такі, як і у моделі bag-of-embeddings. Але як щодо якості?


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_bow = vectorizer.fit_transform(train_data[:TRAIN_SIZE]["text"])

model_tfidf = LogisticRegression(solver='liblinear', C=0.2, penalty="l1")
model_tfidf.fit(X_train_bow, train_data["label"][:TRAIN_SIZE])


X_valid_bow = vectorizer.transform(valid_data["text"])
y_valid_bow = valid_data["label"]
tfidf_acc = model_tfidf.score(X_valid_bow, y_valid_bow)
lab.checkpoint(f"LogReg + TF-IDF accuracy on {TRAIN_SIZE}", tfidf_acc)

# Завдання

Перетренуйте моделі на різних розмірах `TRAIN_SIZE`. Спробуйте кілька значень. Зверніть увагу, як різницю між моделями змінюється в залежності від `TRAIN_SIZE`.


# Embeddings matrix

Досі для доступу до векторів слів ми користувалися бібліотекою `gensim`, яка надавала нам інтерфейс словника (`dict`).

Під капотом, вектори слів зберігаються в одній матриці розмірності $|V| \times d$, де $|V|$ це розмір словника (скільки слів маємо), а $d$ — розмір вектора слова (в цій лабораторній було $d=200$)

In [ ]:
glove.vectors.shape

В моделях глибинного навчання, як правило, справу мають саме з цією embeddings matrix.

Розглянемо два способи отримати вектор потрібного слова з цієї матриці:

## Vector-matrix multiplication

Перший спосіб, це представити слово з індексом $i$ у вигляді one-hot вектора $o_i$. Тоді ембедінг потрібного слова можна отримати в результаті добутку

$$e_i = \text{E}^\intercal o_i $$

In [ ]:
import torch

embeddings_matrix = torch.tensor(glove.vectors)

def embed(token_index: int, embeddings_matrix: torch.tensor) -> torch.tensor:
    vocab_size, embed_dim = embeddings_matrix.shape
    one_hot = torch.zeros(vocab_size)
    one_hot[token_index] = 1
    return one_hot @ embeddings_matrix

assert torch.allclose(
    embed(42, embeddings_matrix),
    torch.tensor(glove.vectors[42]))


In [ ]:
lab.checkpoint("The embedding of one-hot multiplication",
               embed(42, embeddings_matrix).sum())

## nn.Embedding

В PyTorch, як і в більшості deep learning frameworks, є спеціальна функція, яка повертає вектор з потрібним номером: `torch.nn.Embedding`. Вона імплементована більш ефективно, ніж спосіб з vector-matrix multiplication, тож в більшості випадків користуватися варто саме `nn.Embedding`.

In [ ]:
import torch
from torch import nn

embeddings = nn.Embedding(num_embeddings=50_000, embedding_dim=200, _weight=embeddings_matrix)
indexes = torch.LongTensor([42])
embedded = embeddings(indexes)

assert np.isclose(embedded.sum().item(), glove.vectors[42].sum())

In [ ]:
lab.checkpoint("nn.Embeddings", embedded.sum().item())

# Готово!

In [ ]:
lab.answer("ALL DONE! 😊")